# Phase 3: Workforce Intelligence — Step 3.3: Skill Gap Engine

This notebook computes the skill gap for each employee. For every employee, we:
1. Identify their matched role's required skills from `role_skills.csv` (using the O*NET-SOC code mapping).
2. Identify their possessed skills from `employee_skills.csv`.
3. Compute the **Skill Gap** as: `required_skills - possessed_skills`.
4. Assign the **Importance Weight** for each gap skill (from the `Relevance Score` in `role_skills.csv`, which originated from the `Data Value` column in `essential_skills_processed`).
5. Save the output to `data/processed/employee_skill_gaps.csv`.

In [1]:
import os
import pandas as pd
import numpy as np

proc_dir = os.path.join("data", "processed")
print(f"Processed directory: {os.path.abspath(proc_dir)}")

Processed directory: C:\Users\Harshit Mishra\OneDrive\Desktop\enterprise_hr_ai\data\processed


## 1. Load Processed Datasets

In [2]:
df_emp = pd.read_csv(os.path.join(proc_dir, "employees.csv"))
df_emp_skills = pd.read_csv(os.path.join(proc_dir, "employee_skills.csv"))
df_role_skills = pd.read_csv(os.path.join(proc_dir, "role_skills.csv"))

print(f"Employees: {df_emp.shape}")
print(f"Employee possessed skills: {df_emp_skills.shape}")
print(f"Role required skills mapping: {df_role_skills.shape}")

Employees: (1470, 35)
Employee possessed skills: (58417, 3)
Role required skills mapping: (40921, 6)


## 2. Define JobRole to O*NET-SOC Mapping
We use the exact role mapping from Step 1.5.

In [3]:
role_to_code = {
    'Sales Executive': '11-2022.00',
    'Research Scientist': '19-1042.00',
    'Laboratory Technician': '29-2012.00',
    'Manufacturing Director': '11-3051.00',
    'Healthcare Representative': '29-2099.08',
    'Manager': '11-1021.00',
    'Sales Representative': '41-3091.00',
    'Research Director': '11-9121.00',
    'Human Resources': '13-1071.00'
}

print("Job role mapping verified.")

Job role mapping verified.


## 3. Compute Employee-Level Skill Gaps

In [4]:
# Precompute dictionaries for fast lookup
required_skills_dict = {}
for index, row in df_role_skills.iterrows():
    code = row["O*NET-SOC Code"]
    skill = row["Skill Name"]
    score = row["Relevance Score"]
    required_skills_dict[(code, skill)] = score

possessed_skills_dict = df_emp_skills.groupby("employee_id")["current_skill"].apply(set).to_dict()

gap_records = []
for index, row in df_emp.iterrows():
    emp_id = row["EmployeeNumber"]
    job_role = row["JobRole"]
    code = role_to_code[job_role]
    
    # Get all required skills for this role
    role_skills = df_role_skills[df_role_skills["O*NET-SOC Code"] == code]
    req_skills_set = set(role_skills["Skill Name"].tolist())
    
    # Get possessed skills
    pos_skills_set = possessed_skills_dict.get(emp_id, set())
    
    # Gap = Required - Possessed
    gap_skills = req_skills_set - pos_skills_set
    
    for skill in gap_skills:
        score = required_skills_dict.get((code, skill), 0.0)
        gap_records.append({
            "employee_id": emp_id,
            "job_role": job_role,
            "onet_soc_code": code,
            "missing_skill": skill,
            "importance_score": score
        })

df_gap = pd.DataFrame(gap_records)
print(f"Computed {df_gap.shape[0]} individual gap records.")
print(df_gap.head())

Computed 48468 individual gap records.
   employee_id  ... importance_score
0            1  ...              3.0
1            1  ...              3.0
2            1  ...              3.0
3            1  ...              3.0
4            1  ...              3.0

[5 rows x 5 columns]


## 4. Aggregate Skill Gap Statistics
We inspect the overall profile of skill gaps.

In [5]:
avg_gap_size = df_gap.groupby("employee_id")["missing_skill"].count().mean()
print(f"Average missing skills per employee: {avg_gap_size:.2f}")
print(f"Max missing skills: {df_gap.groupby('employee_id')['missing_skill'].count().max()}")
print(f"Min missing skills: {df_gap.groupby('employee_id')['missing_skill'].count().min()}")

Average missing skills per employee: 32.97
Max missing skills: 108
Min missing skills: 7


## 5. Export to CSV

In [ ]:
output_path = os.path.join(proc_dir, "employee_skill_gaps.csv")
df_gap.to_csv(output_path, index=False)
print(f"Successfully saved skill gaps to {output_path}")
print(f"Final schema: {df_gap.columns.tolist()}")
print(f"Total records: {len(df_gap)}")